# 02 - Baseline Forecasts

Naive baselines for 24-hour day-ahead price forecasting, evaluated with a
rolling-origin scheme over the December 2024 test split. These numbers are the
bar any trained model must clear.

In [ ]:
%matplotlib inline

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from energy_price_mlops.data.smard import build_smard_dataset, split_by_timestamp
from energy_price_mlops.eval.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
)
from energy_price_mlops.models.baseline import (
    LastValueForecaster,
    MeanForecaster,
    SeasonalNaiveForecaster,
)

## Load data and locate the test split

The baselines only need the ordered price series. The test split is December
2024; windows before it supply the 168-hour context.

In [ ]:
raw = Path("../data/raw")
dataset = build_smard_dataset(
    actual_consumption_path=raw / "Actual_consumption_202401010000_202501010000_Hour.csv",
    actual_generation_path=raw / "Actual_generation_202401010000_202501010000_Hour.csv",
    day_ahead_prices_path=raw / "Day-ahead_prices_202401010000_202501010000_Hour.csv",
)
splits = split_by_timestamp(
    dataset,
    train_start="2024-01-01",
    valid_start="2024-11-01",
    test_start="2024-12-01",
    test_end="2025-01-01",
)
prices = dataset["day_ahead_price_eur_mwh"].tolist()
test_start = len(splits["train"]) + len(splits["valid"])
print(f"full series: {len(prices):,} hours")
print(f"test split starts at index {test_start:,} ({len(splits['test']):,} hours)")

## Rolling-origin evaluation

For every hour `t` in the test split we take the previous 168 hours as context
and forecast the next 24 hours. Predictions and targets are pooled across all
windows, then scored with MAE, RMSE, and MAPE.

MAPE is reported for completeness but is unreliable here: day-ahead prices pass
through zero and go negative, so percentage errors explode near those hours.
MAE (in EUR/MWh) is the metric to compare.

In [ ]:
CONTEXT = 168
HORIZON = 24

forecasters = {
    "last_value": LastValueForecaster(horizon=HORIZON),
    "mean": MeanForecaster(horizon=HORIZON),
    "seasonal_naive_24h": SeasonalNaiveForecaster(horizon=HORIZON, season_length=24),
}
window_starts = range(test_start, len(prices) - HORIZON + 1)

records = []
for name, forecaster in forecasters.items():
    y_true: list[float] = []
    y_pred: list[float] = []
    for t in window_starts:
        y_true.extend(prices[t : t + HORIZON])
        y_pred.extend(forecaster.predict(prices[t - CONTEXT : t]))
    records.append(
        {
            "model": name,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": root_mean_squared_error(y_true, y_pred),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        }
    )

results = pd.DataFrame(records).set_index("model").round(3)
results

## Results

`seasonal_naive_24h` is the baseline to beat: reusing yesterday's 24-hour
profile captures the daily price shape that `last_value` and `mean` miss. A
trained model is only worth shipping if it improves on this MAE.

In [ ]:
example_t = test_start + 24 * 7  # one week into the test split
target = prices[example_t : example_t + HORIZON]
context = prices[example_t - CONTEXT : example_t]
hours = list(range(HORIZON))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hours, target, marker="o", color="black", label="actual")
for name, forecaster in forecasters.items():
    ax.plot(hours, forecaster.predict(context), marker=".", label=name)
ax.set_title(f"24-hour forecast at test index {example_t}")
ax.set_xlabel("forecast hour")
ax.set_ylabel("EUR/MWh")
ax.legend()
fig.tight_layout()
plt.show()

## Takeaway

These baselines and their test-split MAE are recorded in `docs/model_card.md`.
Phase 2 builds `PriceMLP` and a LightningModule; Phase 4 trains it and must
report an MAE below the `seasonal_naive_24h` figure to justify the model.